## Prepare PPMI data

Created by: Sahana Kowshik

Date: 07/21/2025

Updated: 08/05/2025 (Varuna)

In [1]:
import pandas as pd
import json
import random
import string

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/ppmi"

In [3]:
with open(f"{directory_path}/PPMI_dictionary.json", 'r') as json_file:
    data_dict = json.load(json_file)

In [4]:
ppmi_path = '/projectnb/vkolagrp/datasets/PPMI/csv/raw/PPMI_Curated_Data_Cut_Public_20240729/20240703-Table 1.csv'
ppmi_df = pd.read_csv(ppmi_path, nrows=None)

In [5]:
ppmi_df.shape

(13242, 158)

In [6]:
def get_latest_visits(data):
    result = data.sort_values(by=['PATNO', 'YEAR'], ascending=[True, False])
    result = result.drop_duplicates(subset='PATNO', keep='first').reset_index(drop=True)
    return result

ppmi_latest = get_latest_visits(ppmi_df)
ppmi_latest['COHORT'] = 'PPMI'
# ppmi_latest = ppmi_latest[(~ppmi_latest['cogstate'].isna()) & (~ppmi_latest['PRIMDIAG'].isna())].reset_index(drop=True)

In [7]:
ppmi_latest[['PATNO', 'YEAR']]

,PATNO,YEAR
0,3000,10
1,3001,12
2,3002,11
3,3003,13
4,3004,13
...,...,...
3581,331787,0
3582,333703,0
3583,335170,0
3584,337602,0


# Clean up dictionary
I believe Humza created this data dictionary and recall that for any variables with continuous data with length greater than a certain number, he didn't define a value mapping and described the Values as 'undefined'. I am manually defining 'Type' and

In [8]:
# def find_und_keys_recursively(d, parent_key=""):
#     und_keys = []
#     for key, val in d.items():
#         if isinstance(val, dict):
#             values = val.get("Values")
#             if values and "<und>" in str(values).lower():
#                 full_key = f"{parent_key}.{key}" if parent_key else key
#                 und_keys.append(full_key)
#             else:
#                 und_keys.extend(find_und_keys_recursively(val, parent_key=f"{key}" if parent_key else key))
#     return und_keys

# all_und_keys = find_und_keys_recursively(data_dict)

# print(f"Found {len(all_und_keys)} keys containing '<UND>':")
# cleaned_keys = [k.split('.', 1)[-1] for k in all_und_keys]

# Create JSON

In [9]:
form = {
    'A1': 'Subject Demographics',
    'A3': 'Subject Family History',
    'A4': 'Subject Medications',
    'A5': 'Subject Health History',
    'B1': 'Physical',
    'B3': "Unified Parkinson's Disease Rating Scale (UPDRS)",
    'B6': 'Geriatric Depression Scale (GDS)',
    'B7': 'Activities of Daily Living',
    'B10': 'Sleep Questionnaire',
    'B11': 'Anxiety Questionnaire',
    'B12': "Scales for Outcomes in Parkinson's Disease",
    'C': 'Neuropsychological Battery Summary Scores',
    'G1': 'Genetics',
    'CSF': 'CSF Biomarkers',
    'Blood': 'Blood Biomarkers',
    'Urine': 'Urine Biomarkers',
    'D1': 'Clinician Diagnosis',
    'I': 'Imaging evidence'
}

In [10]:
def collect_skip_cols(d, parent_key=""):
    skip_cols = []
    for key, val in d.items():
        if isinstance(val, dict):
            form_id = val.get("FORM ID")
            val_type = val.get("Type", "").lower() if val.get("Type") else ""
            
            # If this dict has the keys we want, collect
            if form_id in ["U1", "D1"] or val_type == "datetime":
                full_key = f"{parent_key}.{key}" if parent_key else key
                skip_cols.append(full_key)
            else:
                # Recurse into nested dicts
                skip_cols.extend(collect_skip_cols(val, parent_key=f"{parent_key}.{key}" if parent_key else key))
    return skip_cols

skip_cols = collect_skip_cols(data_dict)

print(f"Columns to skip ({len(skip_cols)}):")
print(skip_cols)

Columns to skip (32):
['SITE', 'PATNO', 'COHORT', 'subgroup', 'enroll_phase', 'HIQ_RBD', 'study_status', 'EVENT_ID', 'YEAR', 'visit_date', 'Diagnosis.PRIMDIAG', 'Diagnosis.OTHNEURO', 'Diagnosis.agediag', 'Diagnosis.ageonset', 'Diagnosis.duration', 'Diagnosis.duration_yrs', 'Diagnosis.DOMSIDE', 'Diagnosis.sym_tremor', 'Diagnosis.sym_rigid', 'Diagnosis.sym_brady', 'Diagnosis.sym_posins', 'Diagnosis.sym_other', 'Diagnosis.sym_unknown', 'Diagnosis.MCI_testscores', 'Diagnosis.cogstate', 'Milestones.pm_any', 'Milestones.pm_adl_any', 'Milestones.pm_fd_any', 'Milestones.pm_auto_any', 'Milestones.pm_cog_any', 'Milestones.pm_mc_any', 'Milestones.pm_wb_any']


In [11]:
skip_cols_cleaned = [k.split('.', 1)[-1] for k in skip_cols]

In [12]:
skip_cols_cleaned

['SITE',
 'PATNO',
 'COHORT',
 'subgroup',
 'enroll_phase',
 'HIQ_RBD',
 'study_status',
 'EVENT_ID',
 'YEAR',
 'visit_date',
 'PRIMDIAG',
 'OTHNEURO',
 'agediag',
 'ageonset',
 'duration',
 'duration_yrs',
 'DOMSIDE',
 'sym_tremor',
 'sym_rigid',
 'sym_brady',
 'sym_posins',
 'sym_other',
 'sym_unknown',
 'MCI_testscores',
 'cogstate',
 'pm_any',
 'pm_adl_any',
 'pm_fd_any',
 'pm_auto_any',
 'pm_cog_any',
 'pm_mc_any',
 'pm_wb_any']

In [13]:
# Flatten nested data dictionary for easier lookup
flat_data_dict = {}

def flatten_dict(d, parent_key=''):
    for key, value in d.items():
        if isinstance(value, dict) and 'FORM ID' not in value:
            # This is a nested section, recurse into it
            flatten_dict(value, parent_key)
        elif isinstance(value, dict) and 'FORM ID' in value:
            # This is a field definition
            flat_data_dict[key] = value

flatten_dict(data_dict)

In [14]:
flat_data_dict['SEX']

{'FORM ID': 'A1',
 'Description': 'Sex at birth',
 'Type': 'int',
 'Values': {'1': 'Male', '0': 'Female'}}

In [15]:
datscan_keys = list(data_dict['DATSCAN'].keys())
datscan_keys

['Stage_D',
 'lowput_expected',
 'DATSCAN_CAUDATE_L',
 'DATSCAN_CAUDATE_R',
 'con_caudate',
 'ips_caudate',
 'mean_caudate',
 'DATSCAN_PUTAMEN_L',
 'DATSCAN_PUTAMEN_R',
 'con_putamen',
 'ips_putamen',
 'mean_putamen',
 'con_striatum',
 'ips_striatum',
 'mean_striatum']

In [16]:
def create_mapped_json(row, data_dict, form_dict, skip_cols=None, special_fields=None):
    """
    Maps a data row to structured JSON using a nested data dictionary.
    
    Parameters:
    - row: pandas Series or dict containing the data row
    - data_dict: dictionary containing field mappings, can be nested or flat:
                 {field_name: {'FORM ID': str, 'Description': str, 'Type': str, 'Values': dict/list}}
                 OR {section: {field_name: {...}}}
    - form_dict: dictionary mapping form IDs to form descriptions
    - skip_cols: list of columns to skip (optional)
    - special_fields: dict of special field processing functions (optional)
    
    Returns:
    - JSON string of mapped data
    """
    result = {}
    
    
    
    for column, value in row.items():
        if column in skip_cols:
            continue

         # skip if nan
        if pd.isna(value) or value is None:
            continue
        # # convert to number to check if value's negative
        # try:
        #     if float(value) < 0:
        #         continue  
        # except (ValueError, TypeError):
        #     pass  # Not a number, continue processing
        
        # Skip if column not in flattened data dictionary
        if column not in flat_data_dict:
            continue
        
        # Get form description and create form section if needed
        form_desc = form[flat_data_dict[column]['FORM ID']]
        if form_desc not in result:
            result[form_desc] = {}
            
        # Convert value to appropriate type if specified
        if 'Type' in flat_data_dict[column]:
            try:
                data_type = flat_data_dict[column]['Type'].lower()
                if data_type in ['int', 'integer']:
                    value = int(float(value))
                elif data_type in ['float', 'number']:
                    value = float(value)
                elif data_type in ['str', 'string']:
                    value = str(value)
                elif data_type in ['bool', 'boolean']:
                    value = bool(value)
            except (ValueError, TypeError):
                # If conversion fails, skip this value
                continue
            
        # Get field description
        description = flat_data_dict[column]['Description']
        if column in datscan_keys and 'DATSCAN' not in result[form_desc]:
            result[form_desc]['DATSCAN'] = {}
        
        # Handle value mapping if Values key exists
        if 'Values' in flat_data_dict[column]:
            values_def = flat_data_dict[column]['Values']
            
            # Handle different formats of Values
            if isinstance(values_def, dict):
                # Values is a mapping dict like {"0": "No", "1": "Yes"}
                str_value = str(value)
                if "<UND>: Any text or numbers" in values_def:
                    if column in datscan_keys:
                        result[form_desc]['DATSCAN'][description] = value
                    else:
                        result[form_desc][description] = value
                elif str_value in values_def:
                    mapped_value = values_def[str_value]
                    if mapped_value and mapped_value != ".":  # Skip empty or unknown values
        
                        if column in datscan_keys:
                            result[form_desc]['DATSCAN'][description] = mapped_value
                        else:
                            result[form_desc][description] = mapped_value
                else:
                    continue  # Skip if value doesn't match any in the dictionary
            elif isinstance(values_def, list):
                # Values is a list of allowed values or contains "<UND>: Any text or numbers"
                if any("<UND>" in str(item) for item in values_def):
                    # This means any value is allowed
                    if column in datscan_keys:
                        result[form_desc]['DATSCAN'][description] = value
                    else:
                        result[form_desc][description] = value
                elif value in values_def or float(value) in values_def:
                    # Value is in the allowed list
                    if column in datscan_keys:
                        result[form_desc]['DATSCAN'][description] = value
                    else:
                        result[form_desc][description] = value
                else:
                    continue  # Skip if value not in allowed list
        else:
            # For columns without Values mapping, add value directly
            if column in datscan_keys:
                result[form_desc]['DATSCAN'][description] = value
            else:
                result[form_desc][description] = value
    
    # # Handle special fields if provided - for GAMLSS imaging report 
    # if special_fields:
    #     for field_name, processor_func in special_fields.items():
    #         if field_name in row and pd.notna(row[field_name]):
    #             special_result = processor_func(row[field_name], row)
    #             if special_result:
    #                 result.update(special_result)
    
    # Remove empty form sections
    result = {k: v for k, v in result.items() if v}
    
    # Order results based on form dictionary order
    key_order = [form_desc for form_desc in form_dict.values() if form_desc in result]
    result = {k: result[k] for k in key_order if k in result}
                
    return json.dumps(result)

ppmi_latest['patient_summary'] = ppmi_latest.apply(
    lambda row: create_mapped_json(row, data_dict, form, skip_cols_cleaned), 
    axis=1
)

In [17]:
json.loads(ppmi_latest[ppmi_latest['PATNO'] == 4066].iloc[0]['patient_summary'])

{'Subject Demographics': {'Age at Enrollment': 60.8630136986301,
  'Age at Visit': 60.8630136986301,
  'Sex at birth': 'Male',
  'Years of Education capped at 20': 20,
  'Race': 'White',
  'Indicator for Hispanic/Latino ethnicity': 'No',
  'Handedness': 'Right'},
 'Subject Family History': {'Family History of PD - 3 categories.': 'No Family w/PD',
  'Family History of PD - Binary': 'No Family w/PD'},
 'Subject Medications': {'Is the participant on dopaminergic medication or receiving deep brain stimulation for treating the symptoms of Parkinsons disease?': 'No',
  'Total Levodopa Equivalent Daily Dose': 0.0},
 'Physical': {'Body Mass Index': 25.4716077755434,
  'Indicator of whether systolic blood pressure change is greater or equal to 20 OR diastolic blood pressure change is greater than or equal to 10.': 'No'},
 "Unified Parkinson's Disease Rating Scale (UPDRS)": {'Hoehn & Yahr Stage (includes OFF and untreated scores)': 'Stage 2',
  'Hoehn & Yahr Stage (includes ON and untreated sco

In [18]:
def create_diag_summary(row):
    if 'PRIMDIAG' not in row or pd.isna(row['PRIMDIAG']):
        return json.dumps({})
    
    field_info = flat_data_dict['PRIMDIAG']
    description = field_info['Description']
    value = row['PRIMDIAG']
    
    # Apply value mapping if available
    if 'Values' in field_info:
        int_value = int(float(value))
        str_value = str(int_value)
        if str_value in field_info['Values']:
            mapped_value = field_info['Values'][str_value]
            # print(f"Found mapping: {str_value} -> {mapped_value}")
        else:
            mapped_value = value
            print(f"No mapping found for {str_value}, using original value")
    else:
        mapped_value = value
        print("No Values field found")
    
    result = {description: mapped_value}
    return json.dumps(result)

In [19]:
ppmi_latest['diag_summary'] = ppmi_latest.apply(create_diag_summary, axis=1)

In [20]:
ppmi_latest['diag_summary'][0]

'{"Most likely Primary Diagnosis": "No PD nor other neurological disorder"}'

In [21]:
ppmi_latest[ppmi_latest['PRIMDIAG'].isna()].shape

(43, 160)

In [22]:
ppmi_latest['diag_summary'].value_counts()

diag_summary
{"Most likely Primary Diagnosis": "Idiopathic PD"}                                1506
{"Most likely Primary Diagnosis": "No PD nor other neurological disorder"}         927
{"Most likely Primary Diagnosis": "Prodromal Synucleinopathy"}                     892
{"Most likely Primary Diagnosis": "Other neurological disorder(s)"}                100
{"Most likely Primary Diagnosis": "Essential tremor"}                               44
{}                                                                                  43
{"Most likely Primary Diagnosis": "Prodromal motor PD"}                             21
{"Most likely Primary Diagnosis": "Prodromal non-motor PD"}                         20
{"Most likely Primary Diagnosis": "Dementia with Lewy bodies"}                      10
{"Most likely Primary Diagnosis": "Multiple system atrophy"}                         7
{"Most likely Primary Diagnosis": "Psychogenic parkinsonism"}                        6
{"Most likely Primary Diagnosi

In [23]:
directory_path

'/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/ppmi'

In [24]:
ppmi_latest.to_csv(f'{directory_path}/PPMI_wjson.csv', index=False)

In [25]:
len(ppmi_latest)

3586

# Dropping rows without cogstate or primdiag

In [26]:
# ppmi_latest_filtered = ppmi_latest[~ppmi_latest['cogstate'].isna()].reset_index(drop=True)
# ppmi_latest_filtered = ppmi_latest_filtered[~ppmi_latest_filtered['PRIMDIAG'].isna()].reset_index(drop=True)
# len(ppmi_latest_filtered)

In [27]:
ppmi_latest_filtered = ppmi_latest[(~ppmi_latest['cogstate'].isna()) & (~ppmi_latest['PRIMDIAG'].isna())].reset_index(drop=True)
len(ppmi_latest_filtered)

3315

In [28]:
len(ppmi_latest_filtered)

3315

In [29]:
ppmi_latest_filtered['SEX'].value_counts()

SEX
1    1832
0    1483
Name: count, dtype: int64

In [30]:
ppmi_latest_filtered.to_csv(f'{directory_path}/test.csv', index=False)